# SQL Technical Interview Questions

A beginner-friendly practice set for students, freshers, and internship applicants. The SQL queries are embedded in Python code cells and run against SQLite using Python's built-in `sqlite3` module, so no additional package is required. Run the setup cell first, then run each question cell independently. Expected outputs are shown below each question.

## Sample Database Setup

The `departments` table has one department with no employees. One employee has no assigned department, which gives us useful examples for `LEFT JOIN` and `COALESCE`.

In [2]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")
connection.executescript("""
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    department_id INTEGER,
    salary INTEGER NOT NULL,
    city TEXT NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

INSERT INTO departments (department_id, department_name) VALUES
    (10, 'Engineering'),
    (20, 'HR'),
    (30, 'Sales'),
    (40, 'Finance');

INSERT INTO employees (employee_id, employee_name, department_id, salary, city) VALUES
    (1, 'Alice', 10, 90000, 'New York'),
    (2, 'Bob', 10, 75000, 'San Francisco'),
    (3, 'Carol', 20, 60000, 'New York'),
    (4, 'David', 30, 55000, 'Chicago'),
    (5, 'Eva', 30, 70000, 'San Francisco'),
    (6, 'Frank', NULL, 50000, 'Chicago');
""")

def run_query(sql):
    cursor = connection.execute(sql)
    print(tuple(column[0] for column in cursor.description))
    for row in cursor.fetchall():
        print(row)

print("Sample tables created and records inserted.")

Sample tables created and records inserted.


## Question 1: Select Specific Columns

**Problem:** Display each employee's name and salary.

**Explanation:** `SELECT` chooses columns, and `ORDER BY` makes the result consistently ordered by employee ID.

**Expected output:**
```text
('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Carol', 60000)
('David', 55000)
('Eva', 70000)
('Frank', 50000)
```

In [3]:
query = """
SELECT employee_name, salary
FROM employees
ORDER BY employee_id;
"""
run_query(query)

('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Carol', 60000)
('David', 55000)
('Eva', 70000)
('Frank', 50000)


## Question 2: Filter Rows with WHERE

**Problem:** Find employees whose salary is at least 65,000.

**Explanation:** `WHERE` filters individual rows before the result is returned.

**Expected output:**
```text
('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)
```

In [4]:
query = """
SELECT employee_name, salary
FROM employees
WHERE salary >= 65000
ORDER BY salary DESC;
"""
run_query(query)

('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)


## Question 3: Sort Results with ORDER BY

**Problem:** List all employees from highest to lowest salary.

**Explanation:** `ORDER BY salary DESC` sorts the results in descending salary order.

**Expected output:**
```text
('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)
('Carol', 60000)
('David', 55000)
('Frank', 50000)
```

In [5]:
query = """
SELECT employee_name, salary
FROM employees
ORDER BY salary DESC;
"""
run_query(query)

('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)
('Carol', 60000)
('David', 55000)
('Frank', 50000)


## Question 4: Use Aggregate Functions with GROUP BY

**Problem:** For each department ID, show the employee count and average salary.

**Explanation:** `COUNT` and `AVG` calculate summary values for each group created by `GROUP BY`. The unassigned employee forms a `NULL` department group.

**Expected output:**
```text
('department_id', 'employee_count', 'average_salary')
(None, 1, 50000.0)
(10, 2, 82500.0)
(20, 1, 60000.0)
(30, 2, 62500.0)
```

In [6]:
query = """
SELECT department_id, COUNT(*) AS employee_count, AVG(salary) AS average_salary
FROM employees
GROUP BY department_id
ORDER BY department_id;
"""
run_query(query)

('department_id', 'employee_count', 'average_salary')
(None, 1, 50000.0)
(10, 2, 82500.0)
(20, 1, 60000.0)
(30, 2, 62500.0)


## Question 5: Filter Groups with HAVING

**Problem:** Show departments that have at least two employees and report their average salary.

**Explanation:** `HAVING` filters groups after aggregation; unlike `WHERE`, it can filter using aggregate results.

**Expected output:**
```text
('department_name', 'employee_count', 'average_salary')
('Engineering', 2, 82500.0)
('Sales', 2, 62500.0)
```

In [7]:
query = """
SELECT d.department_name, COUNT(e.employee_id) AS employee_count,
       AVG(e.salary) AS average_salary
FROM departments AS d
JOIN employees AS e ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
HAVING COUNT(e.employee_id) >= 2
ORDER BY d.department_id;
"""
run_query(query)

('department_name', 'employee_count', 'average_salary')
('Engineering', 2, 82500.0)
('Sales', 2, 62500.0)


## Question 6: Combine Tables with INNER JOIN

**Problem:** List employees who have a department, along with the department name.

**Explanation:** An `INNER JOIN` returns rows with matching department IDs in both tables. Frank is excluded because his department ID is `NULL`.

**Expected output:**
```text
('employee_name', 'department_name')
('Alice', 'Engineering')
('Bob', 'Engineering')
('Carol', 'HR')
('David', 'Sales')
('Eva', 'Sales')
```

In [8]:
query = """
SELECT e.employee_name, d.department_name
FROM employees AS e
INNER JOIN departments AS d ON e.department_id = d.department_id
ORDER BY e.employee_id;
"""
run_query(query)

('employee_name', 'department_name')
('Alice', 'Engineering')
('Bob', 'Engineering')
('Carol', 'HR')
('David', 'Sales')
('Eva', 'Sales')


## Question 7: Keep All Rows with LEFT JOIN

**Problem:** Show every department and the number of employees assigned to it, including departments with no employees.

**Explanation:** `LEFT JOIN` preserves all rows from the left (`departments`) table. `COUNT(e.employee_id)` returns zero for departments without a match.

**Expected output:**
```text
('department_name', 'employee_count')
('Engineering', 2)
('HR', 1)
('Sales', 2)
('Finance', 0)
```

In [9]:
query = """
SELECT d.department_name, COUNT(e.employee_id) AS employee_count
FROM departments AS d
LEFT JOIN employees AS e ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY d.department_id;
"""
run_query(query)

('department_name', 'employee_count')
('Engineering', 2)
('HR', 1)
('Sales', 2)
('Finance', 0)


## Question 8: Compare Values with a Subquery

**Problem:** Find employees whose salary is greater than the overall average salary.

**Explanation:** The scalar subquery calculates the average salary, and the outer query selects employees above that value.

**Expected output:**
```text
('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)
```

In [10]:
query = """
SELECT employee_name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC;
"""
run_query(query)

('employee_name', 'salary')
('Alice', 90000)
('Bob', 75000)
('Eva', 70000)


## Question 9: Categorize Values with CASE

**Problem:** Assign each employee a salary category: `High` for salaries of at least 75,000, `Medium` for salaries of at least 60,000, and `Entry` otherwise.

**Explanation:** `CASE` evaluates conditions in order and returns the result for the first matching condition.

**Expected output:**
```text
('employee_name', 'salary_category')
('Alice', 'High')
('Bob', 'High')
('Carol', 'Medium')
('David', 'Entry')
('Eva', 'Medium')
('Frank', 'Entry')
```

In [11]:
query = """
SELECT employee_name,
       CASE
           WHEN salary >= 75000 THEN 'High'
           WHEN salary >= 60000 THEN 'Medium'
           ELSE 'Entry'
       END AS salary_category
FROM employees
ORDER BY employee_id;
"""
run_query(query)

('employee_name', 'salary_category')
('Alice', 'High')
('Bob', 'High')
('Carol', 'Medium')
('David', 'Entry')
('Eva', 'Medium')
('Frank', 'Entry')


## Question 10: Handle NULL with COALESCE

**Problem:** List every employee and their department name. Display `Unassigned` when an employee does not have a department.

**Explanation:** `COALESCE` returns the first non-`NULL` argument. A `LEFT JOIN` ensures the employee without a department is still included.

**Expected output:**
```text
('employee_name', 'department')
('Alice', 'Engineering')
('Bob', 'Engineering')
('Carol', 'HR')
('David', 'Sales')
('Eva', 'Sales')
('Frank', 'Unassigned')
```

In [12]:
query = """
SELECT e.employee_name,
       COALESCE(d.department_name, 'Unassigned') AS department
FROM employees AS e
LEFT JOIN departments AS d ON e.department_id = d.department_id
ORDER BY e.employee_id;
"""
run_query(query)

('employee_name', 'department')
('Alice', 'Engineering')
('Bob', 'Engineering')
('Carol', 'HR')
('David', 'Sales')
('Eva', 'Sales')
('Frank', 'Unassigned')
